# Projeto 1 — Análise de Dados Musicais com Pandas

## Contexto
Você foi contratado pela **Mateus Music** para analisar uma base com músicas, artistas, streams no Spotify e views no YouTube.

Neste notebook, vamos construir uma análise simples e bem guiada para responder perguntas de negócio usando **Pandas**.

## Objetivos da aula
Ao final, você deve conseguir:

1. Carregar uma base de dados no Pandas.
2. Entender a estrutura de um DataFrame.
3. Verificar valores nulos e tipos de dados.
4. Fazer filtros, agrupamentos e ordenações.
5. Criar rankings simples.
6. Gerar gráficos básicos para comunicar resultados.

## 1. Preparação do ambiente

Execute a célula abaixo. Caso use o arquivo `.parquet`, talvez seja necessário instalar o pacote `pyarrow`:

```bash
pip install pandas matplotlib pyarrow
```

Se a sua base estiver em `.csv`, basta deixar o arquivo na mesma pasta do notebook com o nome `Dados_Artistas.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## 2. Carregando os dados

A célula abaixo tenta carregar primeiro o arquivo `Dados_Artistas.parquet`. Se ele não existir, tenta carregar `Dados_Artistas.csv`.

> Se der erro no `.parquet`, instale o `pyarrow` ou use a versão `.csv` da base.

In [ ]:
caminho_parquet = Path('Dados_Artistas.parquet')
caminho_csv = Path('Dados_Artistas.csv')

if caminho_parquet.exists():
    df = pd.read_parquet(caminho_parquet)
elif caminho_csv.exists():
    df = pd.read_csv(caminho_csv)
else:
    raise FileNotFoundError('Coloque Dados_Artistas.parquet ou Dados_Artistas.csv na mesma pasta deste notebook.')

print('Base carregada com sucesso!')
print(f'Linhas: {df.shape[0]}')
print(f'Colunas: {df.shape[1]}')

## 3. Primeira exploração

Antes de analisar qualquer dado, precisamos entender a estrutura da base.

Perguntas iniciais:

- Quantas linhas existem?
- Quais são as colunas?
- Quais tipos de dados aparecem?
- Existem valores nulos?

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum().sort_values(ascending=False).head(20)

## 4. Conferindo as colunas principais

Para este projeto, vamos usar principalmente:

- `Artist`: nome do artista
- `Track`: nome da música
- `Stream`: streams no Spotify
- `Views`: views no YouTube

Execute a célula abaixo para verificar se essas colunas existem.

In [ ]:
colunas_necessarias = ['Artist', 'Track', 'Stream', 'Views']

for coluna in colunas_necessarias:
    if coluna in df.columns:
        print(f'✅ {coluna} encontrada')
    else:
        print(f'❌ {coluna} NÃO encontrada')

## 5. Limpeza simples dos dados

Nesta etapa, vamos criar uma cópia da base apenas com as colunas principais e garantir que `Stream` e `Views` sejam numéricas.

In [ ]:
dados = df[colunas_necessarias].copy()

dados['Stream'] = pd.to_numeric(dados['Stream'], errors='coerce')
dados['Views'] = pd.to_numeric(dados['Views'], errors='coerce')

dados = dados.dropna(subset=['Artist', 'Track', 'Stream', 'Views'])

print('Base após limpeza simples:')
print(dados.shape)
dados.head()

## 6. Quantos artistas diferentes temos?

Aqui usamos:

- `.unique()` para listar valores únicos
- `.nunique()` para contar valores únicos

In [ ]:
artistas_unicos = dados['Artist'].unique()
quantidade_artistas = dados['Artist'].nunique()

print(f'Quantidade de artistas diferentes: {quantidade_artistas}')
print('Primeiros artistas encontrados:')
print(artistas_unicos[:20])

## 7. Top 10 artistas com mais músicas

Agora vamos contar quantas músicas cada artista possui na base.

In [ ]:
top_10_artistas_musicas = (
    dados['Artist']
    .value_counts()
    .head(10)
    .reset_index()
)

top_10_artistas_musicas.columns = ['Artista', 'Quantidade de músicas']
top_10_artistas_musicas

In [ ]:
top_10_artistas_musicas.plot(
    kind='bar',
    x='Artista',
    y='Quantidade de músicas',
    legend=False,
    figsize=(10, 5),
    title='Top 10 artistas com mais músicas na base'
)
plt.xlabel('Artista')
plt.ylabel('Quantidade de músicas')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8. Top 5 músicas com mais views no YouTube

Para encontrar as músicas com mais views, ordenamos a coluna `Views` em ordem decrescente.

In [ ]:
top_5_youtube = (
    dados
    .sort_values('Views', ascending=False)
    .head(5)[['Artist', 'Track', 'Views']]
)

top_5_youtube

## 9. Top 5 artistas com mais streams no Spotify

Aqui usamos `groupby()` para agrupar por artista e depois somamos todos os streams.

In [ ]:
top_5_spotify = (
    dados
    .groupby('Artist', as_index=False)['Stream']
    .sum()
    .sort_values('Stream', ascending=False)
    .head(5)
)

top_5_spotify.columns = ['Artista', 'Total de Streams']
top_5_spotify

In [ ]:
top_5_spotify.plot(
    kind='bar',
    x='Artista',
    y='Total de Streams',
    legend=False,
    figsize=(10, 5),
    title='Top 5 artistas com mais streams no Spotify'
)
plt.xlabel('Artista')
plt.ylabel('Total de Streams')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 10. Comparando Spotify e YouTube

Nem sempre uma música muito forte no Spotify será igualmente forte no YouTube. Vamos comparar as duas métricas.

In [ ]:
amostra = dados[['Track', 'Artist', 'Stream', 'Views']].sort_values('Stream', ascending=False).head(100)

plt.figure(figsize=(8, 5))
plt.scatter(amostra['Stream'], amostra['Views'])
plt.title('Comparação entre Streams no Spotify e Views no YouTube')
plt.xlabel('Streams no Spotify')
plt.ylabel('Views no YouTube')
plt.tight_layout()
plt.show()

## 11. Desafio guiado para os alunos

Responda usando Pandas:

1. Qual artista tem a música com mais views no YouTube?
2. Qual música tem mais streams no Spotify?
3. Qual é a média de streams por música?
4. Qual é a média de views por música?
5. Crie um ranking com os 10 artistas com mais views totais.
6. Crie um gráfico para o ranking anterior.

In [ ]:
# Desafio 1: artista da música com mais views
# Dica: use sort_values('Views', ascending=False).head(1)

In [ ]:
# Desafio 2: música com mais streams
# Dica: use sort_values('Stream', ascending=False).head(1)

In [ ]:
# Desafio 3 e 4: médias
# Dica: use .mean()

In [ ]:
# Desafio 5 e 6: ranking de artistas por views totais
# Dica: groupby('Artist')['Views'].sum()